# Cell 1: Environment Setup & Central Configuration Load

In [2]:
import os
import sys
import yaml
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix

# 1. Dynamically locate the absolute project root folder
current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    PROJECT_ROOT = os.path.abspath(os.path.join(current_dir, ".."))
else:
    PROJECT_ROOT = current_dir

print(f"🌍 Verified Project Root Anchor: {PROJECT_ROOT}")

# 2. Force load configuration
config_file_path = os.path.join(PROJECT_ROOT, "config.yaml")
with open(config_file_path, "r") as f:
    config = yaml.safe_load(f)

print("📝 Central config.yaml successfully mapped.")

🌍 Verified Project Root Anchor: C:\Users\DELL\python_projects\Intrusion-Detection-System-IDS-\intrusion_detection_system
📝 Central config.yaml successfully mapped.


# Cell 2: Load the Processed Testing Dataset

In [3]:
# Construct absolute validation path to the processed test dataset
absolute_test_processed = os.path.join(PROJECT_ROOT, config["data"]["processed_test_path"])

if not os.path.exists(absolute_test_processed):
    raise FileNotFoundError(
        f"❌ Processed test data missing at: {absolute_test_processed}\n"
        "Please run Notebook 2 first to generate your processed feature matrices!"
    )

# Load evaluation matrices
test_df = pd.read_csv(absolute_test_processed)

# Separate features and target labels
X_test = test_df.drop(columns=["target"])
y_test = test_df["target"]

print(f"✔️ Testing dataset successfully loaded. Evaluating against {len(X_test)} data vectors.")

✔️ Testing dataset successfully loaded. Evaluating against 22544 data vectors.


# Cell 3: Native Visualization Helper Function

In [4]:
def plot_confusion_matrix_native(y_true, y_pred, model_title, save_filename):
    """Generates and saves a clean confusion matrix heatmap."""
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(5.5, 4.5))
    sns.heatmap(
        cm, 
        annot=True, 
        fmt="d", 
        cmap="Blues", 
        xticklabels=["Normal", "Anomaly"], 
        yticklabels=["Normal", "Anomaly"]
    )
    plt.ylabel("Actual Infrastructure Status")
    plt.xlabel("Model Inference Prediction")
    plt.title(f"Confusion Matrix: {model_title}")
    
    # Save chart directly into your notebooks directory
    output_chart_path = os.path.join(PROJECT_ROOT, "notebooks", save_filename)
    plt.savefig(output_chart_path, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"📊 Heatmap preserved successfully at: {output_chart_path}")

# Cell 4: Comparative Production Model Evaluation Loop

In [5]:
# Map model keys from config.yaml to user-friendly titles
model_mappings = [
    {"config_key": "rf_path", "name": "Random Forest Classifier", "img_name": "rf_confusion.png"},
    {"config_key": "xgb_path", "name": "XGBoost Classifier", "img_name": "xgb_confusion.png"},
    {"config_key": "svm_path", "name": "Support Vector Machine (SVM)", "img_name": "svm_confusion.png"}
]

performance_benchmarks = []

for entry in model_mappings:
    print(f"\n=======================================================")
    print(f"⚙️ Evaluating Binary Model Target: {entry['name']}")
    print(f"=======================================================")
    
    # Resolve exact absolute path to the .pkl artifact file
    model_absolute_path = os.path.join(PROJECT_ROOT, config["models"][entry["config_key"]])
    
    if not os.path.exists(model_absolute_path):
        print(f"⚠️ Missing binary file payload for {entry['name']} at: {model_absolute_path}")
        print("   Skipping metric parsing for this target model. Run Notebook 3 / train_model.py to generate it.")
        continue
        
    # Deserialization loading sequence from drive into memory
    loaded_clf = joblib.load(model_absolute_path)
    
    # Run test set predictions
    y_pred = loaded_clf.predict(X_test)
    
    # Compute mathematical standard scores
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='binary')
    
    # Log out raw precision/recall engineering breakdowns
    print(classification_report(y_test, y_pred, target_names=["Normal Traffic", "Network Anomaly"], digits=4))
    
    # Save statistics for summary view
    performance_benchmarks.append({
        "Classifier Name": entry["name"],
        "Accuracy Score": f"{accuracy * 100:.2f}%",
        "F1-Score Metrics": f"{f1:.4f}"
    })
    
    # Generate standalone graphical visualization plots
    plot_confusion_matrix_native(y_test, y_pred, entry["name"], entry["img_name"])

# Render dynamic Pandas DataFrame structure summary of final performance
if performance_benchmarks:
    print("\n\n=======================================================")
    print("🏆 SUMMARY PRODUCTION MODEL BENCHMARK BENCH")
    print("=======================================================")
    summary_df = pd.DataFrame(performance_benchmarks)
    display(summary_df)
else:
    print("❌ No models could be loaded. Double-check your models/ directory.")


⚙️ Evaluating Binary Model Target: Random Forest Classifier
⚠️ Missing binary file payload for Random Forest Classifier at: C:\Users\DELL\python_projects\Intrusion-Detection-System-IDS-\intrusion_detection_system\models/random_forest_model.pkl
   Skipping metric parsing for this target model. Run Notebook 3 / train_model.py to generate it.

⚙️ Evaluating Binary Model Target: XGBoost Classifier
⚠️ Missing binary file payload for XGBoost Classifier at: C:\Users\DELL\python_projects\Intrusion-Detection-System-IDS-\intrusion_detection_system\models/xgboost_model.pkl
   Skipping metric parsing for this target model. Run Notebook 3 / train_model.py to generate it.

⚙️ Evaluating Binary Model Target: Support Vector Machine (SVM)
⚠️ Missing binary file payload for Support Vector Machine (SVM) at: C:\Users\DELL\python_projects\Intrusion-Detection-System-IDS-\intrusion_detection_system\models/svm_model.pkl
   Skipping metric parsing for this target model. Run Notebook 3 / train_model.py to gene